### LLM Chain 만들기


## 1. 환경 구성

### 1) 라이브러리 설치

In [ ]:
# uv add python-dotenv langchain langchain-openai

### 2) GROQ 인증키 설정

In [1]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print(GROQ_API_KEY[:3])

gsk


In [2]:
import langchain
print(langchain.__version__)

1.4.3


In [3]:
from langchain_openai import ChatOpenAI

# Groq(OpenAI 호환) 공통 설정 — 모델/온도 변경은 이곳만 수정
GROQ_BASE_URL = "https://api.groq.com/openai/v1"  # Groq API 엔드포인트
GROQ_MODEL = "openai/gpt-oss-120b"  # 대안: "qwen/qwen3.8-27b"
#MOVIE_MODEL = "qwen/qwen3.8-27b"  # 영화 추천 예제용 
TEMPERATURE = 0.7


def get_llm(model: str = GROQ_MODEL, temperature: float = TEMPERATURE) -> ChatOpenAI:
    """Groq 엔드포인트를 사용하는 ChatOpenAI 인스턴스를 생성합니다."""
    return ChatOpenAI(
        api_key=GROQ_API_KEY,
        base_url=GROQ_BASE_URL,
        model=model,
        temperature=temperature,
    )


# 기본 llm (이후 셀에서 재사용)
llm = get_llm()
print(f'model name = {llm.model_name}')

model name = openai/gpt-oss-120b


## 2. LLM Chain

### 1) Prompt + LLM


In [4]:
# model: 공통 설정 셀의 llm 사용

# chain 실행
result = llm.invoke("인공지능 모델의 학습 원리에 대하여 쉽게 설명해 주세요.")
print(type(result))
print(result)

<class 'langchain_core.messages.ai.AIMessage'>
content='## 인공지능 모델이 “배우는” 과정, 쉽게 풀어 설명하면\n\n### 1. 목표는 ‘정답 맞추기’\n- **문제**: 사진을 보면 “고양이”인지 “강아지”인지 맞추는 일, 혹은 문장을 보면 다음에 올 단어를 맞추는 일 등.\n- **정답(라벨)**: 사람이 미리 알려준 답. 예를 들어, 고양이 사진에는 “고양이”라는 라벨이 붙어 있습니다.\n\n### 2. 데이터 → 입력 + 정답\n| 입력 (예시) | 정답(라벨) |\n|-----------|-----------|\n| 고양이 사진 | 고양이 |\n| 강아지 사진 | 강아지 |\n| “오늘은 ___” (문장) | “날씨” |\n\n이런 (입력, 정답) 쌍을 **학습 데이터**라고 부릅니다.\n\n### 3. 모델 = ‘함수(수식)’  \n인공지능 모델은 “입력을 받아서 정답을 예측하는 함수”입니다.  \n- **가벼운 비유**: 입력을 넣으면 자동으로 색을 바꾸는 **색칠기계**.  \n- **파라미터(가중치)**: 색칠기계 안에 있는 조절 가능한 다이얼들. 이 다이얼을 어떻게 돌리느냐에 따라 색칠 결과가 달라집니다.\n\n### 4. 학습 = ‘다이얼을 맞추는 작업’\n1. **예측**: 현재 다이얼(파라미터) 상태로 입력을 넣어 모델이 예측값을 만든다.  \n2. **오차 측정**: 예측값과 실제 정답 사이의 차이를 **손실(Loss)**이라고 부른다.  \n   - 예: “고양이”라고 예측했는데 실제는 “강아지” → 큰 손실.  \n3. **다이얼 조정**: 손실을 줄이기 위해 다이얼을 조금씩 바꾼다.  \n   - 이때 쓰이는 방법이 **경사하강법(Gradient Descent)**.  \n   - “손실이 가장 크게 줄어드는 방향”으로 다이얼을 조금씩 움직인다.  \n4. **반복**: 위 과정을 수천·수만 번 반복하면(에포크) 다이얼이 점점 최적에 가까워져서, 새로운 입력에 대해서도 정답에 가까운

In [5]:
print(result.content)

## 인공지능 모델이 “배우는” 과정, 쉽게 풀어 설명하면

### 1. 목표는 ‘정답 맞추기’
- **문제**: 사진을 보면 “고양이”인지 “강아지”인지 맞추는 일, 혹은 문장을 보면 다음에 올 단어를 맞추는 일 등.
- **정답(라벨)**: 사람이 미리 알려준 답. 예를 들어, 고양이 사진에는 “고양이”라는 라벨이 붙어 있습니다.

### 2. 데이터 → 입력 + 정답
| 입력 (예시) | 정답(라벨) |
|-----------|-----------|
| 고양이 사진 | 고양이 |
| 강아지 사진 | 강아지 |
| “오늘은 ___” (문장) | “날씨” |

이런 (입력, 정답) 쌍을 **학습 데이터**라고 부릅니다.

### 3. 모델 = ‘함수(수식)’  
인공지능 모델은 “입력을 받아서 정답을 예측하는 함수”입니다.  
- **가벼운 비유**: 입력을 넣으면 자동으로 색을 바꾸는 **색칠기계**.  
- **파라미터(가중치)**: 색칠기계 안에 있는 조절 가능한 다이얼들. 이 다이얼을 어떻게 돌리느냐에 따라 색칠 결과가 달라집니다.

### 4. 학습 = ‘다이얼을 맞추는 작업’
1. **예측**: 현재 다이얼(파라미터) 상태로 입력을 넣어 모델이 예측값을 만든다.  
2. **오차 측정**: 예측값과 실제 정답 사이의 차이를 **손실(Loss)**이라고 부른다.  
   - 예: “고양이”라고 예측했는데 실제는 “강아지” → 큰 손실.  
3. **다이얼 조정**: 손실을 줄이기 위해 다이얼을 조금씩 바꾼다.  
   - 이때 쓰이는 방법이 **경사하강법(Gradient Descent)**.  
   - “손실이 가장 크게 줄어드는 방향”으로 다이얼을 조금씩 움직인다.  
4. **반복**: 위 과정을 수천·수만 번 반복하면(에포크) 다이얼이 점점 최적에 가까워져서, 새로운 입력에 대해서도 정답에 가까운 예측을 할 수 있게 된다.

### 5. 핵심 개념을 일상적인 예시로

| 개념 | 일상 비유 |
|------|----------|
| **데이터

### 2) PromptTemplate + LLM

In [6]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("You are an expert in AI Expert. Answer the question.\
                                       <Question>: {input}에 대해 쉽게 설명해주세요.")

print(type(prompt))
print(prompt)

<class 'langchain_core.prompts.prompt.PromptTemplate'>
input_variables=['input'] input_types={} partial_variables={} template='You are an expert in AI Expert. Answer the question.                                       <Question>: {input}에 대해 쉽게 설명해주세요.'


In [7]:
# llm: 공통 설정 셀의 llm 사용
# chain 연결 (LCEL)
chain = prompt | llm
print(type(chain))

# chain 호출
result = chain.invoke({"input": "인공지능 모델의 학습 원리"})
print(type(result))
print(result)

<class 'langchain_core.runnables.base.RunnableSequence'>
<class 'langchain_core.messages.ai.AIMessage'>
content='## 인공지능 모델이 “학습”한다는 것은?\n\n인공지능(특히 딥러닝) 모델은 **‘데이터를 보고 스스로 규칙을 찾아내는 프로그램’**이라고 생각하면 됩니다.  \n우리가 아이에게 “사과는 빨갛고, 바나나는 노랗다”라고 가르치는 것처럼, 모델도 **많은 예시(데이터)를 보여 주면서** 스스로 답을 맞추는 방법을 배우게 됩니다.\n\n아래에서는 가장 흔히 쓰이는 **지도학습**(label이 있는 데이터) 과정을 아주 쉽게 풀어 설명합니다.\n\n---\n\n## 1️⃣ 준비 단계 – 데이터와 목표 정하기  \n\n| 단계 | 내용 | 비유 |\n|------|------|------|\n| **데이터 수집** | 사진, 텍스트, 음성 등 모델이 배울 원본 자료 | 아이에게 그림책을 보여 주는 것 |\n| **정답(라벨) 부여** | 각 데이터에 “고양이”, “강아지” 같은 정답을 붙임 | 그림책에 “이건 고양이야”, “이건 강아지야” 라고 적어 주는 것 |\n| **학습/검증/테스트 셋** | 데이터를 **훈련**(학습), **검증**(조정), **시험**(최종 평가)으로 나눔 | 아이에게 연습문제, 퀴즈, 시험을 따로 내는 것 |\n\n---\n\n## 2️⃣ 모델 안에서 일어나는 일 – ‘예측 → 평가 → 수정’ 루프\n\n### ① 예측 (Prediction)\n- 모델은 현재 가지고 있는 **가중치(파라미터)** 라는 숫자들을 이용해 입력 데이터를 **예측값** 으로 변환합니다.  \n- 예) 사진을 넣으면 “고양이 0.2, 강아지 0.7, 토끼 0.1” 같은 확률을 내놓음.\n\n### ② 평가 (Loss, 손실)\n- 예측값과 실제 정답(라벨) 사이의 차이를 **손실 함수** 로 계산합니다.  \n- 차이가 클수록 “틀렸다!” 라는 신호가 강하게 들어옵

In [8]:
print(result.content)

## 인공지능 모델이 “학습”한다는 것은?

인공지능(특히 딥러닝) 모델은 **‘데이터를 보고 스스로 규칙을 찾아내는 프로그램’**이라고 생각하면 됩니다.  
우리가 아이에게 “사과는 빨갛고, 바나나는 노랗다”라고 가르치는 것처럼, 모델도 **많은 예시(데이터)를 보여 주면서** 스스로 답을 맞추는 방법을 배우게 됩니다.

아래에서는 가장 흔히 쓰이는 **지도학습**(label이 있는 데이터) 과정을 아주 쉽게 풀어 설명합니다.

---

## 1️⃣ 준비 단계 – 데이터와 목표 정하기  

| 단계 | 내용 | 비유 |
|------|------|------|
| **데이터 수집** | 사진, 텍스트, 음성 등 모델이 배울 원본 자료 | 아이에게 그림책을 보여 주는 것 |
| **정답(라벨) 부여** | 각 데이터에 “고양이”, “강아지” 같은 정답을 붙임 | 그림책에 “이건 고양이야”, “이건 강아지야” 라고 적어 주는 것 |
| **학습/검증/테스트 셋** | 데이터를 **훈련**(학습), **검증**(조정), **시험**(최종 평가)으로 나눔 | 아이에게 연습문제, 퀴즈, 시험을 따로 내는 것 |

---

## 2️⃣ 모델 안에서 일어나는 일 – ‘예측 → 평가 → 수정’ 루프

### ① 예측 (Prediction)
- 모델은 현재 가지고 있는 **가중치(파라미터)** 라는 숫자들을 이용해 입력 데이터를 **예측값** 으로 변환합니다.  
- 예) 사진을 넣으면 “고양이 0.2, 강아지 0.7, 토끼 0.1” 같은 확률을 내놓음.

### ② 평가 (Loss, 손실)
- 예측값과 실제 정답(라벨) 사이의 차이를 **손실 함수** 로 계산합니다.  
- 차이가 클수록 “틀렸다!” 라는 신호가 강하게 들어옵니다.  
- 비유: 아이가 문제를 풀고 나서 “오답이야! 정답은 고양이였어” 라고 알려 주는 점수표.

### ③ 수정 (Optimization, 가중치 업데이트)
- **경사하강법(Gradient Descent)** 같은 방법을 써서 손실을 **줄

### 3) PromptTemplate + LLM(invoke()) + StrOutputParser
* invoke() 함수는 동기(synchronous) 방식으로 호출함

In [9]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 컴포넌트 정의 (llm 은 공통 설정 셀의 llm 사용)
prompt = PromptTemplate.from_template("You are an expert in AI Expert. \
                                      Answer the question. <Question>: {input}에 대해 쉽게 설명해주세요.")

output_parser = StrOutputParser()

# 2. chain 생성 (LCEL)
chain = prompt | llm | output_parser
print(type(chain))

# 3. chain의 invoke 호출
result = chain.invoke({"input": "인공지능 모델의 학습 원리"})
print(type(result))
print(result)

<class 'langchain_core.runnables.base.RunnableSequence'>
<class 'langchain_core.messages.base.TextAccessor'>
## 인공지능 모델은 어떻게 “배우는”가요? – 쉬운 설명

### 1. 기본 아이디어: “경험을 통해 개선한다”
인공지능(AI) 모델, 특히 **딥러닝** 모델은 사람의 뇌가 경험을 통해 기억을 쌓고 행동을 바꾸는 방식과 비슷하게 **데이터(경험)** 를 보고 **스스로 규칙을 찾아** 나갑니다.  
- **입력** → **예측** → **정답과 비교** → **오차를 줄이는 방향으로 조정** → 다시 반복  

이 과정을 **학습(Training)** 이라고 합니다.

---

## 2. 학습의 핵심 구성 요소

| 요소 | 설명 (쉽게) | 예시 |
|------|------------|------|
| **데이터** | 모델에게 보여줄 “문제와 정답” 모음 | 사진과 그 사진에 대한 라벨(고양이, 개) |
| **모델(네트워크)** | 입력을 받아서 예측을 내보내는 수학 공식(함수) | 여러 층을 가진 ‘뇌’처럼 생각하면 됨 |
| **손실 함수(Loss)** | 모델이 만든 예측과 실제 정답 사이의 차이를 숫자로 나타낸 것 | “정답과 얼마나 틀렸는가?”를 알려주는 점수 |
| **옵티마이저(Optimizer)** | 손실을 줄이기 위해 모델을 조금씩 바꾸는 방법 | “오차를 줄이려면 어떻게 해야 할까?”를 결정 |
| **학습률(Learning Rate)** | 한 번에 얼마나 크게 바꿀지 정하는 크기 | 너무 크게 하면 놓치고, 너무 작으면 오래 걸림 |

---

## 3. 학습 과정 한 줄 요약

> **“예측 → 오차 확인 → 오차를 줄이도록 조금씩 고치기 → 다시 예측”** 를 여러 번 반복한다.

### 구체적인 단계

1. **입력 데이터를 모델에 넣는다**  
   예) 사진 한 장을 넣으면 모델은 “고양이일 확률 0.6, 개일 확률 0.3, 

### 4) PromptTemplate + LLM(stream()) + StrOutputParser
* stream() 함수는 비동기(asynchronous) 방식으로 호출함

In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 컴포넌트 정의 (llm 은 공통 설정 셀의 llm 사용)
prompt = PromptTemplate.from_template("You are an expert in AI Expert. \
                                      Answer the question. <Question>: {input}에 대해 쉽게 설명해주세요.")

# chain 연결 (LCEL)
chain = prompt | llm | StrOutputParser()

# 스트리밍 출력을 위한 요청
answer = chain.stream({"input": "인공지능 모델의 학습 원리"})
# 스트리밍 출력
#print(answer)

for token in answer:
    # 스트림에서 받은 데이터의 내용을 출력합니다. 줄바꿈 없이 이어서 출력하고, 버퍼를 즉시 비웁니다.
    print(token, end="", flush=True)

### 인공지능 모델, 특히 **머신러닝·딥러닝** 모델이 “학습”하는 원리  
(쉽게 이해할 수 있도록 비유와 단계별 설명을 곁들였습니다)

---

## 1. 기본 아이디어: **경험을 통해 규칙을 찾는다**
- **우리 인간**도 새로운 일을 배울 때, “많이 해보고, 틀리면 고쳐보고” 하는 과정을 거칩니다.  
- **AI 모델**도 비슷합니다. “데이터(경험)”를 많이 보여주고, **예측이 틀릴 때마다 조금씩 고쳐** 나가면서 스스로 규칙(패턴)을 찾아갑니다.

---

## 2. 핵심 구성 요소

| 구성 요소 | 역할 (쉽게 비유) | 예시 |
|----------|----------------|------|
| **데이터** | 학습 재료 (책, 사진, 소리 등) | 고양이와 강아지 사진 모음 |
| **모델(네트워크)** | 머릿속에 있는 ‘수식’ 혹은 ‘뇌’ | 여러 층을 가진 퍼즐 조각들 |
| **손실 함수(Loss)** | 얼마나 틀렸는지 알려주는 점수표 | 시험 점수(오답이 많을수록 점수 낮음) |
| **옵티마이저(Optimizer)** | 틀린 부분을 고쳐주는 도구 | 연필로 틀린 답을 지우고 다시 쓰는 행동 |
| **학습(Training) 과정** | 반복적인 연습 | 매일 숙제 풀기 |

---

## 3. 학습 과정 흐름 (한 번의 반복을 **‘에폭(Epoch)’**이라고 함)

1. **입력 → 예측**  
   - 모델에 **입력 데이터**(예: 고양이 사진)를 넣으면, 현재 모델이 만든 **예측**(예: “강아지일 확률 70%”)이 나옵니다.

2. **손실 계산**  
   - 실제 정답(“고양이”)과 모델의 예측을 비교해 **손실(Loss)**을 구합니다.  
   - 손실이 클수록 “내가 많이 틀렸다”는 뜻입니다.

3. **역전파(Backpropagation)**  
   - 손실을 줄이기 위해 **각 연결(가중치)**이 얼마나 바뀌어야 하는지 계산합니다.  
   - 마치 “어디를 더 열심히 공부해야 할까?”를

##### 2) Multiple Chains
* Multi Chain을 활용한 영화 추천 및 줄거리 요약 

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
prompt1 = ChatPromptTemplate.from_template("{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.")

# Step 2: 추천된 영화의 줄거리를 요약
prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고,\
                                            줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요")

# 영화 예제용 모델 (공통 설정 셀의 get_llm 사용)
llm = get_llm(GROQ_MODEL)

# 체인 1: 영화 추천 (입력: 장르 → 출력: 영화 제목)
chain1 = prompt1 | llm | StrOutputParser()

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
movie = chain1.invoke({"genre": "Drama"})  # 영화 제목 얻기

print(type(movie))
print(" ===> 추천된 영화:")  # movie 값 출력
print(movie)


<class 'langchain_core.messages.base.TextAccessor'>
 ===> 추천된 영화:
**추천 영화: 《소울 (Soul)》 (2020)**  

- **장르**: 드라마·애니메이션  
- **감독**: 피트 닥터(Pete Docter)  
- **주연**: 제이미 폭스(목소리), 티나 페이(목소리)  
- **줄거리**: 재즈 피아니스트가 되기를 꿈꾸던 조 가드너는 뜻밖의 사고로 ‘영혼’ 세계에 떨어지게 된다. 자신의 인생을 되돌아보며 “진정한 삶의 의미”와 “열정”이 무엇인지 찾아가는 과정을 그린 감동적인 이야기입니다.  
- **추천 이유**  
  1. **깊이 있는 메시지** – “삶은 목적이 아니라 순간을 즐기는 것”이라는 사유를 통해 감동을 선사합니다.  
  2. **시각적 아름다움** – 현실 세계와 ‘영혼’ 세계를 색채와 디자인으로 구분해 시각적으로도 뛰어난 작품입니다.  
  3. **전 연령층이 공감** – 인생의 전환점에 서 있는 사람, 꿈을 향해 달리는 청년, 혹은 일상의 소소함을 재조명하고 싶은 모든 이에게 울림을 줍니다.  
  4. **수상 경력** – 아카데미 시상식에서 최우수 애니메이션 영화상, 음악상 등 2관왕을 차지했으며, 비평가들의 호평을 받았습니다.  

> **한 줄 평**: “인생의 작은 순간을 놓치지 말라”는 메시지를 아름다운 애니메이션과 함께 전해 주는, 눈물과 웃음이 교차하는 최고의 드라마 영화입니다.  

**시청 팁**  
- 감상 후에 조 가드너가 말한 “그 순간을 살아라”라는 대사를 떠올리며, 최근에 놓쳤던 일상 속 작은 기쁨을 찾아보세요.  
- 가족·친구와 함께 보면 서로 다른 감동 포인트를 공유하며 대화가 풍성해집니다.  


In [12]:
# 체인 2: 줄거리 요약 (입력: 영화 제목 → 출력: 줄거리)
# prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고,\
#                                             줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요")
chain2 = (
    {"movie": chain1}  # chain1의 출력을 movie 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)
print(chain2)

# 실행: "SF" 장르의 영화 추천 및 줄거리 요약
response = chain2.invoke({"genre": "Drama"})
print("\n🔹 영화 줄거리 요약:\n", response) 

first={
  movie: ChatPromptTemplate(input_variables=['genre'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['genre'], input_types={}, partial_variables={}, template='{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.'), additional_kwargs={})])
         | ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001C808107C80>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001C8081075C0>, root_client=<openai.OpenAI object at 0x000001C8717AF230>, root_async_client=<openai.AsyncOpenAI object at 0x000001C808105880>, model_name='openai/gpt-oss-120b', temperature=0.7, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.groq.com/openai/v1', stream_chunk_timeout=120.0)
         | StrOutputParser()
} middle=[ChatPromptTemplate

##### chain1과 chain2에서 영화 제목이 일관되게 전달 되도록 변경 

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from pprint import pprint

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
prompt1 = ChatPromptTemplate.from_template("{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.")

# Step 2: 추천된 영화의 줄거리를 요약
prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고, 줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요.")

llm = get_llm(GROQ_MODEL)

# 체인 1: 영화 추천 (입력: 장르 → 출력: 영화 제목)
chain1 = prompt1 | llm | StrOutputParser()

# 체인 2: 줄거리 요약 (입력: 영화 제목 → 출력: 줄거리)
chain2 = (
    {"movie": chain1}  # chain1의 출력을 movie 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)

# 실행: "Drama" 장르의 영화 추천 및 줄거리 요약
response = chain2.invoke({"genre": "Drama"})
print("\n🔹 영화 줄거리 요약:")
pprint(response)


🔹 영화 줄거리 요약:
('**그린북 (Green Book) – 2018**  \n'
 '\n'
 '제목: 그린북 (Green Book)  \n'
 '감독: 피터 패럴리 (Peter Farrelly)  \n'
 '캐스팅: 마하샬라 알리 (Mahershala Ali) – 토니 리프, 비고 모텐슨 (Viggo Mortensen) – 토니의 운전사 돈 '
 '셜리  \n'
 '줄거리: 1960년대 인종 차별이 심각했던 미국 남부를 배경으로, 세계적인 흑인 피아니스트 돈 셜리와 그의 백인 운전사 토니 리프가 함께 '
 '남부 투어를 진행하면서 겪는 문화 충돌과 우정을 그린 실화 기반 드라마. 두 사람은 서로 다른 배경과 편견을 넘어 이해와 존중을 배우며, '
 '인간다움과 진정한 우정의 의미를 찾아갑니다.')


In [14]:
# 모델 설정 (공통 설정 셀의 get_llm 사용)
model = get_llm(GROQ_MODEL)

# 멀티 체인 실행 함수
def movie_chain(genre):
    print(f"\n🔹 1단계: '{genre}' 장르 영화 추천")
    
    # 체인 1: 영화 추천
    response1 = model.invoke(f"{genre} 장르에서 유명한 영화 제목 하나만 알려줘")
    movie = response1.content.strip()
    print(f"   → 추천 영화: {movie}")
    
    print(f"\n 2단계: '{movie}' 영화 정보 조회")
    
    # 체인 2: 영화 정보
    response2 = model.invoke(f"{movie} 영화에 대해 간단히 알려줘. 줄거리, 감독, 출연진 포함해서")
    info = response2.content
    print(f"   → 영화 정보:\n{info}")
    
    return movie, info

# 실행
genre = input("원하는 장르를 입력하세요: ")
movie, info = movie_chain(genre)
print(f"\n 최종 결과: '{movie}' 영화가 추천되었습니다!")


🔹 1단계: '액션' 장르 영화 추천
   → 추천 영화: 대표적인 액션 영화 중 하나는 **“다이 하드 (Die Hard)”**입니다.

 2단계: '대표적인 액션 영화 중 하나는 **“다이 하드 (Die Hard)”**입니다.' 영화 정보 조회
   → 영화 정보:
**다이 하드 (Die Hard, 1988)**  

| 항목 | 내용 |
|------|------|
| **감독** | 존 맥티어넌 (John McTiernan) |
| **주연** | 브루스 윌리스 (Bruce Willis) – ‘존 맥클레인’ <br>Alan Rickman – ‘한스 그루버’ <br>Bonnie Bedelia Miller – ‘홀리 맥클레인’ <br>Reginald VelJohnson – ‘서장 알프레드’ |
| **제작·배급** | 20th Century Fox |
| **개봉일** | 1988년 7월 15일 (미국) |
| **장르** | 액션, 스릴러 |

### 간단한 줄거리
- **배경**: 1988년 크리스마스 이브, 로스앤젤레스에 있는 고층 빌딩 ‘Nakatomi Plaza’에서 대규모 기업 파티가 열리고 있다.  
- **주인공**: 뉴욕 경찰관 **존 맥클레인**은 전쟁터에서 복귀한 뒤, 아내 **홀리**와 재결합하기 위해 로스앤젤레스로 온다.  
- **사건**: 파티가 진행되는 도중, 독일계 테러리스트 **한스 그루버** 일당이 건물을 점거하고, 인질들을 잡아 금고를 털려는 계획을 실행한다.  
- **전개**: 건물 안에 남겨진 맥클레인은 무전기와 소수의 무기로 테러리스트들을 하나씩 제압하며, 경찰과 FBI와 협력해 인질들을 구하고 폭탄을 해제하려고 애쓴다.  
- **클라이맥스**: 맥클레인은 건물 옥상에서 그루버와 직접 대치하고, 결국 그루버를 물리쳐 건물을 구한다.  
- **엔딩**: 맥클레인은 부상에도 불구하고 아내와 재회하고, “다음에 또 만나요”라는 대사와 함께 영화가 마무리된다.

### 특징 및 평가
- **액션**: 실감 나는 실내